## 01 HITL Interacción Inicial

##### Human in the loop

In [ ]:
%%pip install -qU langchain langchain-google-genai langchain_community tavily-python aiosqlite langchain-community
%pip install -qU langgraph langgraph-checkpoint-sqlite
%pip install -qU langgraph langgraph-checkpoint-sqlite
%pip install -qU langgraph

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List, Any, Dict
import operator
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage, AIMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_tavily_search import TavilySearch
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from dataclasses import dataclass, field

#### Conectando con la base de datos

In [ ]:
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

conn = sqlite3.connect("checkpoints.db", check_same_thread=False)
memory = SqliteSaver(conn)

#### Creando el estado del agente

In [ ]:
from uuid import uuid4

def reduce_messages(left: list[AnyMessage], right: list[AnyMessage]) -> list[AnyMessage]:

    for message in right:
        if not message.id:
            message.id = str(uuid4())
    
    merged = left.copy()
    for message in right:
        for i, existing in enumerate(merged):
            if existing.id == message.id:
                merged[i] = message
                break
        else:
            merged.append(message)
    return merged

#### Implementando la clase del agente

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], reduce_messages]

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_core.messages import SystemMessage, ToolMessage
from typing import TypedDict, Annotated
from langchain_core.messages import AnyMessage

class Agent:

    def __init__(self, model, tools, system="", checkpointer=None):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_gemini)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(
            checkpointer=checkpointer,
            interrupt_before=["action"] # Añade una interrupción antes de llamar una acción
        )
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)
    
    def call_gemini(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}
    
    def exists_action(self, state: AgentState):
        print(state)
        result = state['messages'][-1]
        return len(result.tool_calls) > 0
    
    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Llamando la herramienta: {t['name']} con los siguientes argumentos: {t['args']}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Retornando al modelo!")
        return {'messages': results}

#### Ejecutando el agente y creando un hilo dinámico

In [ ]:
from datetime import date
current_date = date.today().strftime("%d/%m/%Y")

prompt = f"""Eres un asistente de investigación inteligente y altamente actualizado. \
Tu prioridad principal es encontrar la información más RECIENTE y en TIEMPO REAL siempre que sea posible. \
La fecha actual es {current_date}. \
Al buscar sobre el clima o eventos que se refieran a "hoy" o "ahora", \
DEBES **incluir la fecha actual ({current_date}) en tu consulta a la herramienta de búsqueda**. \
Por ejemplo, si la pregunta es "clima en ciudad x hoy", la consulta para la herramienta debe ser "clima en ciudad x {current_date}". \
Ignora o descarta información que claramente se refiera a fechas pasadas o futuras al responder preguntas sobre el "hoy" o el "ahora". \
Utiliza el mecanismo de búsqueda para buscar información, siempre priorizando el "hoy" o el "ahora". \
Tienes permiso para realizar múltiples llamadas (ya sea de forma conjunta o en secuencia). \
Busca información solo cuando tengas claro lo que deseas. \
Si necesitas investigar alguna información antes de hacer una pregunta de seguimiento, ¡tienes permiso para hacerlo! \
"""

model = ChatGoogleGenerativeAI(model="gemini-1.5-pro")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [ ]:
import uuid

dynamic_thread_id = str(uuid.uuid4())

print(f"Mi nuevo hilo dinámico es: {dynamic_thread_id}")

## 02 HITL Aprobación Humana

In [ ]:
session_id = str(uuid.uuid4())
print(f"DEBUG: Iniciando una nueva conversación con el ID: {session_id}\n")

In [ ]:
user_message = "Cómo está el clima en Pereira hoy?"
messages = [HumanMessage(content=user_message)]
thread_config = {"configurable": {"thread_id": session_id}}

print(f"--- Etapa 1: El agente procesa la entrada y decide la acción ---")
print(f"Usted: {user_message}")

#### Configurando el hilo y procesando la entrada

In [ ]:
for event in abot.graph.stream({"messages": messages}, thread_config):
    for k, v in event.items():
        if k == "llm":
            last_message = v.get('messages', [])[-1]
            if isinstance(last_message, AIMessage) and last_message.tool_calls:
                print(f"\nAgente (decisión): {last_message.tool_calls}")
                print(f"\n--- AGENTE PAUSADO: Intervención humana requerida ---")
            else:
                print(f"\nAgente (respuesta directa/sin tool_calls): {last_message.content}")
                print(f"\n--- AGENTE PAUSADO (respuesta directa, sin acción pendiente) ---")
                current_state = abot.graph.get_state(thread_config)
last_state_message = current_state.values['messages'][-1]

if current_state and current_state.next == ('action',) and isinstance(last_state_message, AIMessage) and last_state_message.tool_calls:
    tool_calls_pending = last_state_message.tool_calls
    if tool_calls_pending:
        print(f"\nEl agente decidió ejecutar la(s) siguiente(s) acción(es) de herramienta:")
        for tc in tool_calls_pending:
            print(f"- Herramienta: {tc['name']}, Argumentos: {tc['args']}")

    user_input = input("\nDesea que el agente ejecute la(s) acción(es)? (si/no): ").lower()

    if user_input == 'si':
        print("\n--- Etapa 2: Retomando la ejecución (Agente ejecutará la acción) ---")
        for event in abot.graph.stream(None, thread_config):
            for k, v in event.items():
                if k == "action":
                    print(f"DEBUG: Herramienta ejecutada y resultado retornado: {v}")
                elif k == "llm":
                    final_response_message = v.get('messages',[])[-1].content
                    print(f"\nAgente (respuesta final): {final_response_message}")
                elif k == "__end__":
                    print(f"DEBUG: El grafo terminó la ejecución.")
        print("\n--- FIN DE LA INTERACCIÓN ---")
    else:
        print("\nEjecución de la acción cancelada por el usuario.")
        print("\n--- FIN DE LA INTERACCIÓN ---")
else:
    print("\nEl agente no decidió ninguna acción de herramienta a pesar de la pausa. Interacción finalizada.")
    if current_state:
        final_response_message = current_state.values['messages'][-1].content
        print(f"Agente (respuesta directa): {final_response_message}")
    print("\n--- FIN DE LA INTERACCIÓN ---")

#### Visualizando el grafo y manejando errores

In [ ]:
from IPython.display import Image, display

print("\n--- Tratando genterar el PNG del Grafo via Mermaid ---")
try:
    image_data = abot.graph.get_graph().draw_mermaid_png()
    display(Image(data=image_data))
except AttributeError:
    print("Método .draw_mermaid_png() no encontrado o sin soporte.")
    print("Tratando generar únicamente el código Mermaid...")
    try:
        mermaid_code = abot.graph.get_graph().draw_mermaid()
        print(f"\n--- Código Mermaid Generado (Pegado en https://mermaid.live/) ---")
        print(mermaid_code)
    except Exception as e_mermaid:
        print(f"Error al generar el código Mermaid: {e_mermaid}")
except Exception as e:
    print(f"Error inesperado al tratar generar el grafo: {e}")

#### Iniciando una nueva interacción y evaluando el estado 

In [ ]:
from IPython.display import Image, display
import uuid

new_session_id = str(uuid.uuid4())
print(f"DEBUG: Iniciando una nueva conversación con ID: {new_session_id}\n")
new_user_message = "Cuál es la distancia entre CDMX y Tokio?"
new_messages = [HumanMessage(content=new_user_message)]
new_thread_config = {"configurable": {"thread_id": new_session_id}}

print("--- Iniciando Nueva Interacción: Agente procesa la entrada y recibe la acción ---")
print(f"Usted: {new_user_message}")
print(f"DEBUG: ID del nuevo hilo: {new_session_id}")

print("\n--- Agente pensando y pausando ---")
try:
    for event in abot.graph.stream({"messages": new_messages}, new_thread_config):
        for k, v in event.items():
            if k == "llm":
                if v and 'messages' in v and v['messages']:
                    llm_message_from_event = v['messages'][0]
                    if hasattr(llm_message_from_event, 'tool_calls') and llm_message_from_event.tool_calls:
                        print(f"nAgente (decisión): {llm_message_from_event.tool_calls}")
                        print("\n--- AGENTE PAUSADO: Intervención humana necesaria ---")
                    
                    elif llm_message_from_event.content:
                        print(f"\nAgente (respuesta directa): {llm_message_from_event.content}")
                        print("\n--- AGENTE NO PAUSÓ POR LA HERRAMIENTA (Respuesta directa de la LLM) ---")

except Exception as e:
    print(f"DEBUG: Stream interrumpido como esperado: {e}")
current_state_snapshot = abot.graph.get_state(new_thread_config)

if current_state_snapshot:
    print(f"\nDEBUG: Estado actual obtenido para el nuevo ID del hilo: {new_session_id}")
    
    snapshot_thread_id = None
    snapshot_thread_ts = None
    
    if hasattr(current_state_snapshot, 'config') and isinstance(current_state_snapshot.config, dict):
        if 'configurable' in current_state_snapshot.config and isinstance(current_state_snapshot.config['configurable'], dict):
            if 'thread_id' in current_state_snapshot.config['configurable']:
                snapshot_thread_id = current_state_snapshot.config['configurable']['thread_id']
            if '__run_id' in current_state_snapshot.config['configurable']:
                snapshot_thread_ts = current_state_snapshot.config['configurable']['__run_id']
            elif 'thread_ts' in current_state_snapshot.config['configurable']:
                snapshot_thread_ts = current_state_snapshot.config['configurable']['thread_ts']
                
    if snapshot_thread_id is None:
        snapshot_thread_id = new_session_id

    print(f"DEBUG: ID del Hilo (del snapshot): {snapshot_thread_id}")
    print(f"DEBUG: Timestamp del snapshot (thread_ts): {snapshot_thread_ts}")
    print(f"DEBUG: Mensajes en el snapshot (al momento de la pausa): {current_state_snapshot.values.get('messages')}")

    if current_state_snapshot.values and 'messages' in current_state_snapshot.values:
        last_msg_in_snapshot = current_state_snapshot.values['messages'][-1]
        print(f"DEBUG: Tipo del último mensaje en el snapshot para la inyección: {type(last_msg_in_snapshot)}")
        if hasattr(last_msg_in_snapshot, 'tool_calls') and last_msg_in_snapshot.tool_calls:
            print(f"DEBUG: Último mensaje en el snapshot CONTIENE tool_calls. LISTO PARA LA INYECCIÓN!")
        else:
            print(f"DEBUG: Último mensaje en el snapshot NO CONTIENE tool_calls o está vacío. ¡PROBLEMA EN LA PAUSA!")
        if current_state_snapshot.next != ():
            print(f"\n--- Agente pausado y listo para la intervención. ---")
        else:
            print(f"\n--- ATENCIÓN: El agente NO está pausado donde debería. El grafo puede haber finalizado. ---")
else:
    print(f"DEBUG: Ningún estado encontrado para el nuevo ID de hilo: {new_session_id}. Verifique la configuración del hilo o si el agente pausó.") 